# Zipf's Law

**Computational Text Analysis Pack · Module 02 — Describing a Corpus**

*Technique:* Rank-frequency distributions; the long tail and what it costs you

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/02-describing-corpora/02b_zipfs_law.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Word frequencies are not spread out evenly, and they are not bell-shaped. They
follow a very steep curve: the commonest word is roughly twice as common as the
second, three times the third, and so on. That is **Zipf's law**, and it holds in
every language anyone has checked.

Two consequences you will meet constantly:

- A handful of words account for a third of your corpus
- **Most of your vocabulary appears once** — and single occurrences cannot support
  a statistical claim

## The tooling

**`collections.Counter`** is the whole engine of frequency analysis — a
standard-library dictionary that counts. **matplotlib** draws the distributions.
The field's dedicated tools (**AntConc**, **#LancsBox**) wrap the same counting
in a GUI.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
LANGUAGE = "Hausa"          # ← change this

sub = df[df["language"] == LANGUAGE]
counts = frequencies(sub["text"])
total = sum(counts.values())
ranked = counts.most_common()

print(f"{LANGUAGE}: {total:,} tokens, {len(counts):,} types")
print()
for k in (1, 10, 100, 1000):
    covered = sum(n for _, n in ranked[:k])
    print(f"  top {k:>5} types cover {covered / total:>6.1%} of all tokens")

## The curve

Plotted on ordinary axes it looks like a cliff. Plotted with both axes on a log
scale, Zipf's law becomes a straight line — which is what makes it a *law* rather
than an observation.

In [ ]:
ranks = np.arange(1, len(ranked) + 1)
freqs = np.array([n for _, n in ranked])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(ranks[:400], freqs[:400])
axes[0].set_xlabel("rank"); axes[0].set_ylabel("frequency")
axes[0].set_title("linear axes: a cliff")

axes[1].loglog(ranks, freqs)
axes[1].set_xlabel("rank (log)"); axes[1].set_ylabel("frequency (log)")
axes[1].set_title("log-log axes: a straight line")
fig.suptitle(f"Zipf's law in {LANGUAGE} news")
plt.tight_layout(); plt.show()

In [ ]:
# The tail, which is where most of your vocabulary lives.
from collections import Counter as _C
by_count = _C(n for _, n in ranked)
print(f"{'appears':>9}  {'how many types':>15}  {'% of vocabulary':>16}")
print("-" * 45)
for k in (1, 2, 3, 4, 5):
    print(f"{k:>9}× {by_count[k]:>15,}  {by_count[k] / len(counts):>15.1%}")
rare = sum(v for k, v in by_count.items() if k <= 2)
print()
print(f"{rare / len(counts):.0%} of the vocabulary appears once or twice.")
print("These are hapax legomena. They are most of your word list and almost")
print("none of your text — and no test can tell a real pattern from noise in them.")

### 🔧 Your turn 1

Change `LANGUAGE` to English, then to Swahili, and re-run.

The log-log line has roughly the same slope in all of them. Does the *proportion*
of once-only words differ? Morphologically rich languages — where one root takes
many affixes — produce more distinct forms and therefore a fatter tail.

## Why this constrains your analysis

Zipf is the reason for the `min_count` and `min_df` arguments you will see in
every later notebook.

- **04a keyness** ignores words below a minimum count — with one occurrence, the
  test statistic is meaningless
- **06/07 vectorising** sets `min_df` — a word in one document adds a column of
  zeros and no information
- **08 topic modelling** is dominated by frequent words unless you weight against
  them

Those thresholds are not tidiness. They are the only defence against reporting a
finding that rests on a single occurrence.

In [ ]:
# What a min_count threshold actually removes.
for m in (1, 2, 5, 10, 25):
    kept = sum(1 for _, n in ranked if n >= m)
    tokens_kept = sum(n for _, n in ranked if n >= m)
    print(f"min_count ≥ {m:>3}: {kept:>7,} types kept "
          f"({kept / len(counts):>5.1%} of vocabulary) "
          f"covering {tokens_kept / total:>5.1%} of tokens")
print()
print("Note the asymmetry: a threshold of 5 throws away most of the vocabulary")
print("and almost none of the text. That is Zipf's law working in your favour.")

### 🔧 Your turn 2

Which threshold would you defend in a paper? Write the sentence you would put in
your methods section, with the number and the reason.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The slope is remarkably stable across languages — that is the
point of Zipf. What differs is the tail: Swahili and Hausa, with heavier
morphology, produce more once-only forms than English, because *walked*,
*walking* and *walks* are separate types before any lemmatisation. For African
languages with no lemmatiser available, that fat tail is permanent, and it is
another reason to prefer methods robust to it.

**Your turn 2.** Something like: *"Types occurring fewer than five times in the
corpus were excluded (n = …,
covering under 2% of tokens), since single and
double occurrences cannot support the log-likelihood test used below."* The
number matters less than stating it and giving the reason.

</details>

## Where this stops

You know the shape of a frequency distribution and why thresholds exist. Next:
**02c — type/token ratio**, a measure of vocabulary richness with a trap in it.